**📝 Step 1: Create the Dataset in Colab**

Instead of uploading a CSV, run this cell to generate raw_skills.csv directly in your Colab environment. This ensures the code runs immediately without file-path errors.

In [4]:
# Cell 1: Generate the Dataset
import pandas as pd

data = {
    'job_role': [
        'Data Scientist', 'DevOps Engineer', 'Backend Developer',
        'Cloud Architect', 'Frontend Developer', 'Machine Learning Engineer',
        'Data Analyst', 'Cybersecurity Analyst'
    ],
    'skills': [
        'Python, SQL, Machine Learning, Data Analysis, Statistics',
        'AWS, Docker, Kubernetes, CI/CD, Linux, Automation',
        'Java, Python, SQL, APIs, Data Structures, Git',
        'Cloud Computing, AWS, Automation, Python, Networking',
        'JavaScript, React, HTML, CSS, Web Design',
        'Python, TensorFlow, PyTorch, Machine Learning, Data Analysis',
        'SQL, Excel, Python, Data Visualization, Statistics',
        'Network Security, Python, Linux, Risk Analysis, Cryptography'
    ]
}

df = pd.DataFrame(data)
df.to_csv('raw_skills.csv', index=False)
print("✅ Dataset 'raw_skills.csv' created successfully in Colab!")
df.head()

✅ Dataset 'raw_skills.csv' created successfully in Colab!


,job_role,skills
0,Data Scientist,"Python, SQL, Machine Learning, Data Analysis, ..."
1,DevOps Engineer,"AWS, Docker, Kubernetes, CI/CD, Linux, Automation"
2,Backend Developer,"Java, Python, SQL, APIs, Data Structures, Git"
3,Cloud Architect,"Cloud Computing, AWS, Automation, Python, Netw..."
4,Frontend Developer,"JavaScript, React, HTML, CSS, Web Design"


**⚙️ Step 2: The Recommendation Engine (4-Step Pipeline)**

Copy and paste this into the next cell. It implements the exact Ingestion → Scoring → Sorting → Filtering pipeline mandated by DecodeLabs, including the Cold Start bypass (requiring ≥ 3 skills).

In [8]:
# Cell 2: AI Recommendation Logic (FIXED)
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_data(filepath):
    # Step 1: Ingestion (Load Data)
    return pd.read_csv(filepath)

def build_recommender(df):
    # Step 2: Processing (TF-IDF Vectorization)
    # FIX: 'skills' is already a string. TfidfVectorizer handles commas perfectly.
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['skills'])
    return vectorizer, tfidf_matrix

def recommend(user_skills, df, vectorizer, tfidf_matrix, top_n=3):
    # Cold Start Bypass: Enforce minimum 3 skills
    if len(user_skills) < 3:
        return "⚠️ Cold Start Alert: Please provide at least 3 skills for accurate matching."

    # Transform user input into the same vector space
    user_input_str = " ".join(user_skills)
    user_vector = vectorizer.transform([user_input_str])

    # Step 3: Scoring (Cosine Similarity)
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Add scores and Sort (Step 4: Sorting & Filtering)
    df['similarity_score'] = similarity_scores
    sorted_df = df.sort_values(by='similarity_score', ascending=False)

    # Filter Top-N
    return sorted_df[['job_role', 'skills', 'similarity_score']].head(top_n)

# --- Execution ---
print("🚀 DecodeLabs Project 3: Tech Stack Recommender")
print("------------------------------------------------")

# Load the generated data
data = load_data('raw_skills.csv')
vectorizer, tfidf_matrix = build_recommender(data)

# Onboarding Survey Simulation (Cold Start Bypass)
user_input = input("👉 Enter at least 3 skills separated by commas (e.g., Python, Cloud, Automation): ")
user_skills_list = [skill.strip() for skill in user_input.split(',')]

# Get recommendations
recommendations = recommend(user_skills_list, data, vectorizer, tfidf_matrix, top_n=3)

print("\n🎯 Top 3 Recommended Career Paths:")
for index, row in recommendations.iterrows():
    print(f"🔹 {row['job_role']} (Match: {row['similarity_score']:.2%})")
    print(f"   Required Skills: {row['skills']}\n")

🚀 DecodeLabs Project 3: Tech Stack Recommender
------------------------------------------------
👉 Enter at least 3 skills separated by commas (e.g., Python, Cloud, Automation): Python, AWS, Automation

🎯 Top 3 Recommended Career Paths:
🔹 Cloud Architect (Match: 59.62%)
   Required Skills: Cloud Computing, AWS, Automation, Python, Networking

🔹 DevOps Engineer (Match: 44.19%)
   Required Skills: AWS, Docker, Kubernetes, CI/CD, Linux, Automation

🔹 Data Scientist (Match: 9.95%)
   Required Skills: Python, SQL, Machine Learning, Data Analysis, Statistics

